# Quotes Scraper

This notebook demonstrates how to scrape quotes from [Quotes to Scrape](http://quotes.toscrape.com/) - a website specifically designed for practicing web scraping.

**Features covered:**
- Basic HTML parsing
- Extracting multiple elements
- Handling pagination
- Exporting to CSV

In [1]:
from bs4 import BeautifulSoup
import requests
import pandas as pd
import time

## Single Page Scraping
Let's start by scraping quotes from a single page.

In [2]:
url = "http://quotes.toscrape.com/"

response = requests.get(url)
print(f"Status Code: {response.status_code}")

soup = BeautifulSoup(response.text, 'html.parser')

Status Code: 200


In [3]:
# Find all quote containers
quotes = soup.find_all('div', class_='quote')
print(f"Found {len(quotes)} quotes on this page")

# Extract data from first quote as example
first_quote = quotes[0]
print("\nFirst Quote:")
print(f"Text: {first_quote.find('span', class_='text').text}")
print(f"Author: {first_quote.find('small', class_='author').text}")
print(f"Tags: {[tag.text for tag in first_quote.find_all('a', class_='tag')]}")

Found 10 quotes on this page

First Quote:
Text: “The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”
Author: Albert Einstein
Tags: ['change', 'deep-thoughts', 'thinking', 'world']


In [4]:
# Extract all quotes from the page
quotes_data = []

for quote in quotes:
    text = quote.find('span', class_='text').text.strip('""')
    author = quote.find('small', class_='author').text
    tags = ', '.join([tag.text for tag in quote.find_all('a', class_='tag')])

    quotes_data.append({
        'quote': text,
        'author': author,
        'tags': tags
    })

df = pd.DataFrame(quotes_data)
df.head()

,quote,author,tags
0,“The world as we have created it is a process ...,Albert Einstein,"change, deep-thoughts, thinking, world"
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling,"abilities, choices"
2,“There are only two ways to live your life. On...,Albert Einstein,"inspirational, life, live, miracle, miracles"
3,"“The person, be it gentleman or lady, who has ...",Jane Austen,"aliteracy, books, classic, humor"
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe,"be-yourself, inspirational"


## Multi-Page Scraping with Pagination
Now let's scrape all pages by following the pagination.

In [5]:
def scrape_all_quotes():
    """Scrape all quotes from all pages."""
    base_url = "http://quotes.toscrape.com"
    all_quotes = []
    page = 1

    while True:
        url = f"{base_url}/page/{page}/"
        response = requests.get(url)
        soup = BeautifulSoup(response.text, 'html.parser')

        quotes = soup.find_all('div', class_='quote')

        # Break if no quotes found (end of pages)
        if not quotes:
            break

        for quote in quotes:
            text = quote.find('span', class_='text').text.strip('""')
            author = quote.find('small', class_='author').text
            tags = ', '.join([tag.text for tag in quote.find_all('a', class_='tag')])

            all_quotes.append({
                'quote': text,
                'author': author,
                'tags': tags,
                'page': page
            })

        print(f"Scraped page {page} - Total quotes: {len(all_quotes)}")
        page += 1

        # Be respectful - add delay between requests
        time.sleep(1)

    return all_quotes

# Scrape all quotes
all_quotes = scrape_all_quotes()

Scraped page 1 - Total quotes: 10
Scraped page 2 - Total quotes: 20
Scraped page 3 - Total quotes: 30
Scraped page 4 - Total quotes: 40
Scraped page 5 - Total quotes: 50
Scraped page 6 - Total quotes: 60
Scraped page 7 - Total quotes: 70
Scraped page 8 - Total quotes: 80
Scraped page 9 - Total quotes: 90
Scraped page 10 - Total quotes: 100


In [6]:
# Create DataFrame and display summary
df_all = pd.DataFrame(all_quotes)
print(f"Total quotes scraped: {len(df_all)}")
print(f"\nUnique authors: {df_all['author'].nunique()}")
print(f"\nTop 5 authors by quote count:")
print(df_all['author'].value_counts().head())

Total quotes scraped: 100

Unique authors: 50

Top 5 authors by quote count:
author
Albert Einstein    10
J.K. Rowling        9
Marilyn Monroe      7
Dr. Seuss           6
Mark Twain          6
Name: count, dtype: int64


In [7]:
# Export to CSV
df_all.to_csv('quotes_output.csv', index=False)
print("Data exported to quotes_output.csv")

Data exported to quotes_output.csv
